# 🛸 Dell Inspiron 5301 Custom Linux ISO - Google Colab Hub
Directly assemble or compile the single **full, un-split 3.72 GB ISO** straight into your **5 TB Google Drive**!

---
### 🚀 Step 1: Connect Your Google Drive
Run the cell below and authorize Google Drive access.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
print("✅ 5 TB Google Drive mounted successfully at /content/drive/MyDrive!")

---
### ⚡ Step 2: Instant Transfer to Google Drive (Under 1 Minute!)
This fetches the compiled parts from the cloud runner and reassembles them into a **single, full 3.72 GB ISO file** directly in your Google Drive folder ().

In [ ]:
import os

out_dir = "/content/drive/MyDrive/CustomOS"
os.makedirs(out_dir, exist_ok=True)
final_iso = os.path.join(out_dir, "DellOS-5301-20261028-amd64.iso")

print("⬇️ Downloading Part 1 (1.88 GB)...")
!curl -L "https://github.com/vishxtr/dell-5301-linux/releases/download/v1.0.15/DellOS-5301-20261028-amd64.iso.part-aa" -o /content/part-aa

print("⬇️ Downloading Part 2 (1.84 GB)...")
!curl -L "https://github.com/vishxtr/dell-5301-linux/releases/download/v1.0.15/DellOS-5301-20261028-amd64.iso.part-ab" -o /content/part-ab

print("🧩 Assembling single full ISO into your Google Drive...")
!cat /content/part-aa /content/part-ab > "{final_iso}"
!rm -f /content/part-aa /content/part-ab

print("🔍 Verifying single ISO in Google Drive:")
!ls -lh "{final_iso}"
print(f"
🎉 SUCCESS! Full single ISO is saved at: {final_iso}")

---
### 🛠️ Optional: Compile From Scratch Inside Colab
If you wish to compile the entire ISO completely from source inside Colab:

In [ ]:
# Install tools
!sudo apt-get update
!sudo apt-get install -y git make gettext po4a debian-archive-keyring debootstrap squashfs-tools xorriso isolinux syslinux-efi grub-pc-bin grub-efi-amd64-bin mtools dosfstools

# Upstream live-build
!rm -rf /tmp/live-build
!git clone --depth 1 https://salsa.debian.org/live-team/live-build.git /tmp/live-build
!cd /tmp/live-build && sudo make install

# Clone custom OS repository
!rm -rf /content/dell-5301-linux
!git clone https://github.com/vishxtr/dell-5301-linux.git /content/dell-5301-linux

# Configure
%cd /content/dell-5301-linux
!mkdir -p build
%cd /content/dell-5301-linux/build
!sudo lb config \n    --mode debian \n    --distribution bookworm \n    --parent-distribution bookworm \n    --mirror-bootstrap "http://deb.debian.org/debian/" \n    --mirror-chroot "http://deb.debian.org/debian/" \n    --mirror-binary "http://deb.debian.org/debian/" \n    --parent-mirror-bootstrap "http://deb.debian.org/debian/" \n    --parent-mirror-chroot "http://deb.debian.org/debian/" \n    --parent-mirror-binary "http://deb.debian.org/debian/" \n    --archive-areas "main contrib non-free non-free-firmware" \n    --parent-archive-areas "main contrib non-free non-free-firmware" \n    --architectures amd64 \n    --linux-flavours amd64 \n    --binary-images iso-hybrid \n    --bootloader grub-efi \n    --memtest none \n    --system live \n    --security false \n    --apt-indices false \n    --apt-recommends true \n    --iso-application "DellOS-5301" \n    --iso-publisher "Custom OS Builder" \n    --iso-volume "DellOS-5301-20261028"

!sudo cp -r ../config/package-lists/* config/package-lists/
!sudo cp -r ../config/includes.chroot/* config/includes.chroot/
!sudo cp -r ../config/hooks/* config/hooks/

# Build
!sudo lb build

# Save single ISO directly to Google Drive
!mkdir -p /content/drive/MyDrive/CustomOS
!cp *.iso /content/drive/MyDrive/CustomOS/DellOS-5301-compiled.iso
!ls -lh /content/drive/MyDrive/CustomOS/DellOS-5301-compiled.iso
print("🎉 Compiled single ISO is saved directly in Google Drive!")